# Matrix Factorisations

A factorisation writes a matrix as a product of simpler ones. This notebook computes the LU, Cholesky and QR factorisations of the same matrix, checks each by multiplying back, and shows why a factorisation is computed once and then reused.

In [ ]:
Pollis.packages("Plots")

In [ ]:
using LinearAlgebra, Plots

In [ ]:
A = [4.0 1 2; 1 5 1; 2 1 6]   # symmetric, with a dominant diagonal
(symmetric = issymmetric(A), positive_definite = isposdef(A))

## LU: elimination, recorded

$PA = LU$, with $L$ lower triangular (the multipliers of Gaussian elimination), $U$ upper triangular and $P$ the row swaps.

In [ ]:
F = lu(A)
(L = F.L, U = F.U, row_order = F.p, check = norm(F.L * F.U - A[F.p, :]))

The determinant is the product of the diagonal of $U$, up to the sign of the row swaps.

In [ ]:
(product_of_pivots = prod(diag(F.U)), determinant = det(A))

## Cholesky: for symmetric positive definite matrices

$A = R^\top R$ with $R$ upper triangular. It needs half the work of LU, and it exists only when $A$ is positive definite, so trying it is the cheapest test.

In [ ]:
Ch = cholesky(Symmetric(A))
R = Ch.U
(R = R, check = norm(transpose(R) * R - A))

A use: draw correlated normal vectors. If $z$ has independent standard normal entries, $R^\top z$ has covariance $A$.

In [ ]:
using Random
rng = Xoshiro(42)
Z = randn(rng, 3, 100_000)
X = transpose(R) * Z
sample_cov = X * transpose(X) / size(X, 2)
round.(sample_cov, digits=2)

## QR: orthogonal times triangular

$A = QR$, with $Q$ orthogonal ($Q^\top Q = I$) and $R$ upper triangular. It works for any shape and is the stable way to do least squares.

In [ ]:
Fq = qr(A)
Q = Matrix(Fq.Q)
(check = norm(Q * Fq.R - A), orthogonality = norm(transpose(Q) * Q - I))

## Factorise once, solve many times

The factorisation costs about $n^3$ operations, each later solve about $n^2$. With a larger matrix the difference is plain.

In [ ]:
n = 500
rng = Xoshiro(1)
B = randn(rng, n, n)
M = B * transpose(B) + n * I   # symmetric positive definite
rhs = randn(rng, n, 100)       # 100 right-hand sides
Fm = cholesky(Symmetric(M))
solve_reusing(F, rhs) = [F \ rhs[:, j] for j in 1:size(rhs, 2)]
solve_fresh(M, rhs) = [M \ rhs[:, j] for j in 1:size(rhs, 2)]
solve_reusing(Fm, rhs); solve_fresh(M, rhs)   # compile first
(seconds_reusing = @elapsed(solve_reusing(Fm, rhs)), seconds_refactoring = @elapsed(solve_fresh(M, rhs)))

## References

[Householder (1958)](https://doi.org/10.1145/320941.320947), Unitary triangularization of a nonsymmetric matrix, *Journal of the ACM* 5(4), 339–342.

[Trefethen and Bau (1997)](https://doi.org/10.1137/1.9780898719574), *Numerical Linear Algebra*, SIAM.